In [ ]:
import time as _t0
NB_T0 = _t0.time()                 # notebook start (R6's 8.3 h deadline counts from here)
# ===== Step 0: check every input before any inference stage starts (fails in seconds, not hours) =====
import json as _json
from pathlib import Path as _Path

def _checked_mount(slug, required):
    root = _Path('/kaggle/input')
    candidates = [root / slug, root / 'competitions' / slug]
    candidates += list(root.glob('datasets/*/' + slug)) + list(root.glob('datasets/*/' + slug + '/*'))
    candidates += list(root.glob('*/' + slug)) + list(root.glob(slug + '/*'))
    found = list(dict.fromkeys(p for p in candidates if (p / required).is_file()))
    if len(found) != 1:
        raise RuntimeError('Attach the required dataset %s; found %s' % (slug, found))
    return found[0]

_bundle = _checked_mount('rsna-knee-r6-frozen-fe055406-w1', 'manifest.json')
_manifest = _json.loads((_bundle / 'manifest.json').read_text())
if _manifest.get('bundle_id') != 'fe055406fad6a12f':
    raise RuntimeError('Use the R6 bundle fe055406fad6a12f; attached bundle: %s' % _manifest.get('bundle_id'))
_listed = sorted(set(_manifest.get('sha256', {})) | set(_manifest.get('models', [])) | set(_manifest.get('models2') or []))
_missing = [name for name in _listed if not (_bundle / name).is_file()]
if _missing:
    raise FileNotFoundError('Missing bundle files: %s' % _missing)
_checked_mount('rsna-orthofoundation-fold0-epoch7-20261004', 'receipt.json')   # R6's decode code + pinned wheels
def _find_files(name, max_depth=6):
    # walk /kaggle/input but never the competition's DICOM tree (hundreds of thousands of files)
    import os as _os
    hits = []
    for _root, _dirs, _files in _os.walk('/kaggle/input'):
        if 'rsna-knee-abnormality-detection' in _root or _root.count('/') - 3 >= max_depth:
            _dirs[:] = []
            continue
        _dirs[:] = [d for d in _dirs if d != 'rsna-knee-abnormality-detection']
        if name in _files:
            hits.append(_root + '/' + name)
    return hits

_checked_mount('rsna-knee-r6-src', 'probe_dual.py')   # R6's worker scripts
_checked_mount('rsna-knee-abnormality-detection', 'test.csv')
# our leg's inputs: all 14 models must be attached (this notebook is mostly ours, so a missing model stops it here)
_ours_models = [p for p in _find_files('model.pt') if 'a50v2-full' in p]
_reader = _find_files('cnxt_v0_fold0.pt')
print('INPUTS_READY bundle=fe055406fad6a12f models=%d files=%d | our full-data models %d (expect 14) | reader %s'
      % (len(_manifest.get('models', [])), len(_listed), len(_ours_models), 'ok' if _reader else 'MISSING'), flush=True)
if len(_ours_models) != 14:
    raise RuntimeError('our leg needs exactly 14 full-data models (found %d)' % len(_ours_models))
if not _reader:
    raise RuntimeError('attach goodpjw2008/rsna-knee-2-5d-convnext-reader (our leg installs its JPEG decoders + timm wheel)')


In [ ]:
%%writefile /kaggle/working/our_leg.py
# ===== Our leg: 14 full-data ConvNeXt-T models (8 general + 6 single-finding specialists) -> _ours.csv =====
# Reads every test study's DICOMs, builds the same input as training (cache 0266fa68 settings, dense5 layout,
# stride-2 three-slice windows, left knees mirrored), runs all models on one GPU and writes per-finding ranks.
# Settings come from the checkpoints; anything this script does not support stops it with an assertion.
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")

import json, re, time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata

cv2.setNumThreads(0)

T0 = time.time()
OWNER = "harrisonhoran"                                          # only our own weights datasets are opened
EXPECT_RUNS = int(os.environ.get("OUR_LEG_EXPECT_RUNS", "14"))  # 8 general + 6 specialists
SPEC_W = float(os.environ.get("OUR_LEG_SPEC_W", "0.5"))         # specialists' share of their own column
DEADLINE = float(os.environ.get("OUR_LEG_DEADLINE", "inf"))     # absolute epoch seconds
BATCH = 8                                                       # studies per GPU batch (55 windows each)
CHUNK = 64                                                      # studies built on the CPU per round
WORKERS = max(1, os.cpu_count() or 4)
WORK = Path("/kaggle/working")
LEG_OUT = WORK / "_ours.csv"
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEV.type == "cuda":
    torch.backends.cudnn.benchmark = True                       # fixed input shapes: let cuDNN pick its fastest kernels once

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]


def log(msg):
    print(f"[our leg {(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)


def walk_files(pred, base="/kaggle/input"):
    """Files under /kaggle/input matching `pred(name)`, never crawling the DICOM folders."""
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        for f in files:
            if pred(f):
                yield Path(root) / f


def find_root():
    for p in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection")]:
        if (p / "test.csv").exists():
            return p
    for p in walk_files(lambda f: f == "sample_submission.csv"):
        if (p.parent / "test.csv").exists():
            return p.parent
    raise FileNotFoundError("competition data not attached")


ROOT = find_root()
test = pd.read_csv(ROOT / "test.csv")
sample = pd.read_csv(ROOT / "sample_submission.csv")
assert list(sample.columns) == ["StudyInstanceUID"] + TARGETS, f"unexpected submission columns: {list(sample.columns)}"
log(f"device {DEV} | test {len(test):,} studies | deadline in {(DEADLINE - time.time()) / 60:.0f} min")

# ---------------------------------------------------------------- checkpoints: one full-data model per run
CKPTS = {}
for p in sorted(walk_files(lambda f: f.endswith(".pt"))):
    if OWNER not in p.parts and p.name != "model.pt":           # other .pt files (R6 etc.) are never opened
        continue
    try:
        ck = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e:
        log(f"skip {p}: {type(e).__name__}")
        continue
    if not (isinstance(ck, dict) and "cache_key" in ck and "model" in ck and "cfg" in ck):
        continue                                                # not one of our training checkpoints
    cfg = ck["cfg"]
    if not (cfg["backbone"].startswith("convnext") and "study_layers" in cfg and cfg.get("fold") == "all"):
        continue                                                # only full-data ConvNeXt CNN-MIL models
    assert cfg["layout"] == "dense5" and int(cfg.get("overlap_stride") or 0) == 2, f"{p}: layout {cfg['layout']}"
    assert cfg.get("laterality") == "flip_left_to_right", f"{p}: laterality {cfg.get('laterality')}"
    assert cfg["study_layers"] == 0, f"{p}: study transformer layers {cfg['study_layers']}"
    key = f"{ck['cache_key']}/{cfg['backbone']}@{cfg['img']}/{cfg.get('name', '?')}"
    if key in CKPTS and not ck["val_auc"] > CKPTS[key][1]["val_auc"]:
        log(f"duplicate run {key}: dropping {p}")               # same run attached twice: keep the better copy
        continue
    if key in CKPTS:
        log(f"duplicate run {key}: dropping {CKPTS[key][0]}")
    CKPTS[key] = (p, ck)
RUN_KEYS = sorted(CKPTS)
assert len(RUN_KEYS) == EXPECT_RUNS, f"expected {EXPECT_RUNS} runs, found {len(RUN_KEYS)}: {RUN_KEYS}"

ref = CKPTS[RUN_KEYS[0]][1]
for key in RUN_KEYS:                                           # every run must see exactly the same input
    ck = CKPTS[key][1]
    for k in ["cache_key", "slots", "slot_sizes", "preproc", "targets"]:
        assert ck[k] == ref[k], f"run {key}: '{k}' differs from run {RUN_KEYS[0]}"
    assert ck["cfg"]["img"] == ref["cfg"]["img"], f"run {key}: input size differs"
assert ref["targets"] == TARGETS
PREPROC = dict(ref["preproc"])                                 # cache 0266fa68 settings
IMG = int(ref["cfg"]["img"])                                   # model input size (256)
RUN_FOCUS = {k: list(CKPTS[k][1]["cfg"].get("focus_labels") or []) if CKPTS[k][1]["cfg"].get("focus") else []
             for k in RUN_KEYS}
for k in RUN_KEYS:
    assert set(RUN_FOCUS[k]) <= set(TARGETS), (k, RUN_FOCUS[k])
log(f"{len(RUN_KEYS)} runs | general {sum(not v for v in RUN_FOCUS.values())} | specialists "
    f"{ {k.split('/')[-1]: v for k, v in RUN_FOCUS.items() if v} }")

# ---------------------------------------------------------------- layout: slots and stride-2 windows (as in training)
SLOTS = list(ref["slots"])                                     # series-type codes, e.g. "11S" = fluid, fat-sat, sagittal
N_CACHE = PREPROC["n_cache"]                                   # 24 slices kept per series
SLOT_PLANES = [s[-1] for s in SLOTS]
assert set(SLOT_PLANES) <= {"S", "C", "A"}, SLOTS
CEN = np.arange(1, N_CACHE - 1, 2)                             # window centres 1, 3, ..., 21 -> 11 windows per series
TAKE = np.concatenate([k * N_CACHE + np.concatenate([[c - 1, c, c + 1] for c in CEN]) for k in range(len(SLOTS))])
SLOT_SIZES = list(ref["slot_sizes"])
assert SLOT_SIZES == [3 * len(CEN)] * len(SLOTS), f"slot sizes {SLOT_SIZES} do not match stride-2 windows"
BOUNDS = np.cumsum([0] + SLOT_SIZES)                           # where each slot starts in a run's own volume
TSLOT = torch.tensor(np.repeat(np.arange(len(SLOTS)), [n // 3 for n in SLOT_SIZES]))   # window -> slot
log(f"slots {SLOTS} | {len(TSLOT)} windows per study")


def series_table(csv_path, series_dir):
    s = pd.read_csv(csv_path)
    s["code"] = (s["Fluid_Sensitive"].astype(int).astype(str)
                 + s["Fat_Suppression"].astype(int).astype(str)
                 + s["Anatomical_Plane"].str[0])
    s["dir"] = [str(Path(series_dir) / a / b) for a, b in zip(s["StudyInstanceUID"], s["SeriesInstanceUID"])]
    s["n_files"] = [sum(1 for e in os.scandir(d) if e.name.endswith(".dcm")) if os.path.isdir(d) else 0
                    for d in s["dir"]]
    return s[s["n_files"] > 0].reset_index(drop=True)


def assign_series(g):
    """One series per slot: the series with that exact code, most files first; None if the study has none."""
    used, out = set(), []
    for code in SLOTS:
        c = g[(g["code"] == code) & ~g["SeriesInstanceUID"].isin(used)].sort_values("n_files", ascending=False)
        uid = c["SeriesInstanceUID"].iloc[0] if len(c) else None
        used.add(uid)
        out.append(uid)
    return out


# ---------------------------------------------------------------- preprocessing (identical to the cache notebook)
ORDER_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]
SIDE_TAGS = ORDER_TAGS + ["PixelSpacing", "Rows", "Columns", "Laterality", "ImageLaterality", "SeriesDescription",
                          "ProtocolName", "StudyDescription", "BodyPartExamined"]
_LEFT = re.compile(r"(?:\bLT\b|\bLEFT\b|\bLINKS?\b|\bIZQ|\bGAUCHE\b|\bSINIST|\bSOL\b|\bLEWY|\bLEWE|\bVASEN)", re.I)
_RIGHT = re.compile(r"(?:\bRT\b|\bRIGHT\b|\bRECHTS?\b|\bDERECH|\bDROIT|\bDESTR|\bPRAW|\bOIKEA)", re.I)


def side_info(h):
    """Knee-side evidence from one header, same rules as study_knee_side.csv (training):
    tag = Laterality / ImageLaterality (L/R); text = descriptions say left xor right; cx = slice-centre x (LPS, + = left)."""
    info = {"tag": "", "text": "", "cx": None}
    if h is None:
        return info
    for t in ("Laterality", "ImageLaterality"):
        v = str(h.get(t, "") or "").strip().upper()[:1]
        if v in ("L", "R"):
            info["tag"] = v
            break
    txt = " | ".join(str(h.get(t, "") or "") for t in ("SeriesDescription", "ProtocolName", "StudyDescription",
                                                         "BodyPartExamined"))
    l, r = bool(_LEFT.search(txt)), bool(_RIGHT.search(txt))
    info["text"] = "L" if l and not r else ("R" if r and not l else "")
    try:
        ipp = np.asarray([float(v) for v in h.ImagePositionPatient], float)
        iop = np.asarray([float(v) for v in h.ImageOrientationPatient], float)
        ps = [float(v) for v in h.PixelSpacing]
        rows, cols = int(h.Rows), int(h.Columns)
        info["cx"] = float((ipp + iop[:3] * ps[1] * (cols - 1) / 2 + iop[3:] * ps[0] * (rows - 1) / 2)[0])
    except Exception:
        pass
    return info


def study_side(infos):
    """Study side from its series' evidence: tag/text per series (mode over series), else median x beyond 20 mm."""
    votes = [i["tag"] or i["text"] for i in infos if i and (i["tag"] or i["text"])]
    if votes:
        nl, nr = votes.count("L"), votes.count("R")
        return "L" if nl >= nr else "R"                 # ties -> L, like pandas mode()[0] in training
    xs = [i["cx"] for i in infos if i and i["cx"] is not None]
    if xs:
        cx = float(np.median(xs))
        return "L" if cx > 20 else ("R" if cx < -20 else "?")
    return "?"


def flip_left(vol):
    """Left knee -> right-knee orientation, as in training (on the windowed volume):
    coronal/axial mirrored left<->right, sagittal slice order reversed."""
    parts = []
    for k, plane in enumerate(SLOT_PLANES):
        seg = vol[BOUNDS[k]:BOUNDS[k + 1]]
        parts.append(seg[::-1] if plane == "S" else seg[..., ::-1])
    return np.ascontiguousarray(np.concatenate(parts))


def order_files(series_dir):
    """-> (files in slice order, side evidence from the middle file)."""
    files = sorted(e.path for e in os.scandir(series_dir) if e.name.endswith(".dcm"))
    geo, inst, mid_h = [], [], None
    for j, f in enumerate(files):
        g = i = None
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True,
                                specific_tags=SIDE_TAGS if j == len(files) // 2 else ORDER_TAGS)
            if j == len(files) // 2:
                mid_h = h
            if "ImageOrientationPatient" in h and "ImagePositionPatient" in h:
                iop = np.asarray(h.ImageOrientationPatient, float)
                ipp = np.asarray(h.ImagePositionPatient, float)
                g = float(ipp @ np.cross(iop[:3], iop[3:]))
            if "InstanceNumber" in h and h.InstanceNumber is not None:
                i = float(h.InstanceNumber)
        except Exception:
            pass
        geo.append(g)
        inst.append(i)
    info = side_info(mid_h)
    if files and all(v is not None for v in geo):
        return [f for _, f in sorted(zip(geo, files))], info
    if files and all(v is not None for v in inst):
        return [f for _, f in sorted(zip(inst, files))], info
    return files, info


def pick_indices(n, k, span):
    if n == 1:
        return np.zeros(k, int)
    lo = int(round(span[0] * (n - 1)))
    hi = int(round(span[1] * (n - 1)))
    if hi <= lo:
        return np.full(k, n // 2)
    return np.linspace(lo, hi, k).round().astype(int)


def read_slice(path, invert_mono1):
    ds = pydicom.dcmread(path, force=True)
    a = ds.pixel_array.astype(np.float32)
    if a.ndim == 3:
        a = a[..., 0] if a.shape[-1] in (3, 4) else a[0]
    if invert_mono1 and str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a
    return a, ds


def tissue_center(a):
    lo, hi = np.percentile(a, [1, 99])
    u8 = (np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
    u8 = cv2.GaussianBlur(u8, (5, 5), 0)
    _, m = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, _, stats, _ = cv2.connectedComponentsWithStats(m)
    if n <= 1:
        return None
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = stats[k, :4]
    if w * h < 0.05 * a.size:
        return None
    return ((y + h / 2) / a.shape[0], (x + w / 2) / a.shape[1])


def crop_mm(a, spacing, crop_mm_, center=None):
    h, w = a.shape
    if not spacing or spacing <= 0:
        side = min(h, w)
    else:
        side = min(int(round(crop_mm_ / spacing)), h, w)
    if PREPROC.get("center") == "tissue":
        cy, cx = (center[0] * h, center[1] * w) if center else (h / 2, w / 2)
        y0 = int(round(min(max(cy - side / 2, 0), h - side)))
        x0 = int(round(min(max(cx - side / 2, 0), w - side)))
    else:
        y0, x0 = (h - side) // 2, (w - side) // 2
    return a[y0:y0 + side, x0:x0 + side]


def normalize_stack(slices, pct):
    px = np.concatenate([s[::2, ::2].ravel() for s in slices])
    lo, hi = np.percentile(px, pct)
    return [np.clip((s - lo) / max(hi - lo, 1e-6), 0, 1) for s in slices]


def to_uint8(s, img):
    interp = cv2.INTER_AREA if s.shape[0] >= img else cv2.INTER_LINEAR
    s = cv2.resize(s, (img, img), interpolation=interp)
    return (s * 255 + 0.5).astype(np.uint8)


def process_series(series_dir):
    """-> (uint8 (n_cache, img, img), side evidence) or (None, evidence)."""
    P = PREPROC
    files, info = order_files(series_dir)
    if not files:
        return None, info
    idx = pick_indices(len(files), P["n_cache"], P["span"])
    raw = {}
    for i in sorted(set(idx.tolist())):
        try:
            a, ds = read_slice(files[i], P["invert_mono1"])
        except Exception:
            continue
        ps = getattr(ds, "PixelSpacing", None)
        raw[i] = (a, float(ps[0]) if ps is not None else None)
    if not raw:
        return None, info
    center = None
    if P.get("center") == "tissue":
        mid = sorted(raw)[len(raw) // 2]
        center = tissue_center(raw[mid][0])
    crops = {i: crop_mm(a, sp, P["crop_mm"], center) for i, (a, sp) in raw.items()}
    good = sorted(crops)
    stack = [crops[min(good, key=lambda g: abs(g - i))] for i in idx]
    stack = normalize_stack(stack, P["pct"])
    return np.stack([to_uint8(s_, P["img"]) for s_ in stack]), info


def build_study(task):
    """(uid, [series dir or None per slot]) ->
    (uid, uint8 (n_slots * n_cache, H, W), presence (n_slots,), ok, knee side 'L'/'R'/'?')."""
    uid, dirs = task
    parts, present, infos = [], [], []
    try:
        for d in dirs:
            arr = None
            if d is not None:
                try:
                    arr, info = process_series(d)
                    infos.append(info)
                except Exception:
                    arr = None
            parts.append(arr if arr is not None else np.zeros((N_CACHE, PREPROC["img"], PREPROC["img"]), np.uint8))
            present.append(1.0 if arr is not None else 0.0)
        return uid, np.concatenate(parts), np.array(present, np.float32), sum(present) > 0, study_side(infos)
    except Exception:
        return uid, None, None, False, "?"


te = series_table(ROOT / "test_series.csv", ROOT / "test_series")
dmap = dict(zip(te["SeriesInstanceUID"], te["dir"]))
PLAN = {st: assign_series(g) for st, g in te.groupby("StudyInstanceUID")}
tasks = [(u, [dmap[s] if s else None for s in PLAN[u]]) for u in test["StudyInstanceUID"] if u in PLAN]
log(f"{len(tasks):,} of {len(test):,} test studies have usable series")


# ---------------------------------------------------------------- model (same as training)
class TimmCNNEncoder(nn.Module):
    """Same wrapper as the training notebook (weights under enc.m.*)."""

    def __init__(self, name):
        super().__init__()
        import timm
        self.m = timm.create_model(name, pretrained=False, num_classes=0, global_pool="avg")
        self.num_features = self.m.num_features

    def forward(self, x):
        return self.m(x.contiguous(memory_format=torch.channels_last))


class KneeNet(nn.Module):
    """ConvNeXt-T per window -> + slot and position embeddings -> per-finding attention pooling -> 12 logits.
    Missing series are masked out of the attention."""

    def __init__(self, n_out, n_slots, triplet_slot, cfg):
        super().__init__()
        self.enc = TimmCNNEncoder(cfg["backbone"]).to(memory_format=torch.channels_last)
        d = self.enc.num_features
        self.register_buffer("triplet_slot", triplet_slot.clone())
        self.slot_emb = nn.Parameter(torch.zeros(n_slots, d))
        self.pos_emb = nn.Parameter(torch.zeros(len(triplet_slot), d))
        self.norm = nn.LayerNorm(d)
        self.att = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, n_out))
        self.cls_w = nn.Parameter(torch.randn(n_out, d) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        self.register_buffer("prior", torch.zeros(n_out, n_slots))     # all zeros; kept so the checkpoints load strictly

    def forward(self, x, present):
        B, T = x.shape[:2]
        f = self.enc(x.flatten(0, 1)).view(B, T, -1)
        f = self.norm(f + self.slot_emb[self.triplet_slot] + self.pos_emb)
        keep = present[:, self.triplet_slot] > 0
        a = self.att(f) + self.prior[:, self.triplet_slot].T.unsqueeze(0)
        a = a.float().masked_fill(~keep.unsqueeze(-1), -1e4).softmax(dim=1)
        pooled = torch.einsum("btn,btd->bnd", a, f.float())
        return (pooled * self.cls_w).sum(-1) + self.cls_b


MODELS = []
for key in RUN_KEYS:
    p, ck = CKPTS[key]
    m = KneeNet(len(TARGETS), len(SLOTS), TSLOT, ck["cfg"])
    m.load_state_dict(ck["model"], strict=True)
    MODELS.append(m.to(DEV).eval())
del CKPTS
log(f"loaded {len(MODELS)} models on {DEV}" + (f" ({torch.cuda.get_device_name(DEV)})" if DEV.type == "cuda" else ""))

MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 1, 3, 1, 1)


def to_model_input(x_uint8):
    """uint8 (B, 3 * windows, H, W) -> normalised float (B, windows, 3, IMG, IMG) on the GPU."""
    x = x_uint8.to(DEV, non_blocking=True).float().div_(255.0)
    if x.shape[-1] != IMG:
        x = F.interpolate(x, size=(IMG, IMG), mode="bilinear", align_corners=False, antialias=True)
    B = x.shape[0]
    x = x.view(B, x.shape[1] // 3, 3, IMG, IMG)
    return (x - MEAN) / STD


@torch.no_grad()
def predict_part(part):
    """One batch of built studies -> (n_models, B, 12) probabilities. One shared input for all models;
    fp16 on GPU, with an fp32 retry."""
    vols = [flip_left(r[1][TAKE]) if r[4] == "L" else r[1][TAKE] for r in part]
    x = to_model_input(torch.from_numpy(np.stack(vols)))
    p = torch.from_numpy(np.stack([r[2] for r in part])).to(DEV)
    outs = []
    for m in MODELS:
        try:
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
                z = m(x, p).float()
            if not torch.isfinite(z).all():
                z = m(x, p).float()
        except RuntimeError:
            torch.cuda.empty_cache()
            z = m(x, p).float()
        outs.append(torch.sigmoid(z).cpu().numpy())
    return np.stack(outs)


# ---------------------------------------------------------------- predict the test set
preds, n_fail, n_left, n_unknown, t_start = {}, 0, 0, 0, time.time()
with ProcessPoolExecutor(WORKERS) as ex:
    for c0 in range(0, len(tasks), CHUNK):
        if time.time() > DEADLINE:
            log(f"DEADLINE reached at {c0:,}/{len(tasks):,} studies; leg abandoned")
            raise SystemExit(1)
        results = list(ex.map(build_study, tasks[c0:c0 + CHUNK]))
        n_left += sum(1 for r in results if r[3] and r[4] == "L")
        n_unknown += sum(1 for r in results if r[3] and r[4] == "?")
        ok = [r for r in results if r[3]]
        n_fail += len(results) - len(ok)
        for b in range(0, len(ok), BATCH):
            part = ok[b:b + BATCH]
            try:
                out = predict_part(part)
                for j, r in enumerate(part):
                    preds[r[0]] = out[:, j]
            except Exception as e:                     # e.g. out of GPU memory at 8: retry in halves of 4
                torch.cuda.empty_cache() if DEV.type == "cuda" else None
                log(f"batch of {len(part)} failed ({type(e).__name__}: {e}); retrying in halves")
                for h in (part[:len(part) // 2], part[len(part) // 2:]):
                    if not h:
                        continue
                    try:
                        out = predict_part(h)
                        for j, r in enumerate(h):
                            preds[r[0]] = out[:, j]
                    except Exception as e2:
                        n_fail += len(h)
                        log(f"half batch failed ({type(e2).__name__}: {e2})")
        done = min(c0 + CHUNK, len(tasks))
        el = time.time() - t_start
        log(f"{done:,}/{len(tasks):,} studies | eta {(len(tasks) - done) * el / max(done, 1) / 60:.0f} min")

log(f"predicted {len(preds):,} studies; {n_fail} failed; {len(test) - len(tasks)} had no usable series | "
    f"knee side: {n_left:,} left (mirrored), {n_unknown:,} unknown")

# ---------------------------------------------------------------- per-finding ranks: general models, then specialists
ids = [u for u in test["StudyInstanceUID"] if u in preds]
cover = len(ids) / max(len(test), 1)
if not (ids and cover >= 0.97):
    log(f"NOT written: coverage {cover:.1%}")
    raise SystemExit(1)
A = np.nan_to_num(np.stack([preds[u] for u in ids], axis=1), nan=0.5)            # (n_models, n, 12)
R = np.stack([[rankdata(A[m, :, j]) / len(ids) for j in range(len(TARGETS))]
              for m in range(A.shape[0])])                                      # (n_models, 12, n)
gen = [i for i, k in enumerate(RUN_KEYS) if not RUN_FOCUS[k]]
assert gen, "no general (all-finding) model loaded"
ranked = R[gen].mean(axis=0).T                                                  # (n, 12): general models, all columns
for j, t in enumerate(TARGETS):                                                 # specialists: only their own column
    sp = [i for i, k in enumerate(RUN_KEYS) if t in RUN_FOCUS[k]]
    if sp:
        s = R[sp, j].mean(axis=0)
        log(f"specialists for {t}: {len(sp)} run(s), weight {SPEC_W} | rank corr with general "
            f"{np.corrcoef(ranked[:, j], s)[0, 1]:.3f}" if len(ids) > 2 else f"specialists for {t}: {len(sp)}")
        ranked[:, j] = (1 - SPEC_W) * ranked[:, j] + SPEC_W * s
o = pd.DataFrame(ranked, columns=TARGETS)
o.insert(0, "StudyInstanceUID", ids)
tmp = WORK / "_ours.tmp.csv"
o.to_csv(tmp, index=False)
os.replace(tmp, LEG_OUT)
log(f"wrote {LEG_OUT} ({len(ids):,}/{len(test):,} studies)")


In [ ]:
# ===== Our leg first: install its decoders / timm privately, then build + predict every test study on the GPU =====
import glob as _c_glob, os as _c_os, subprocess as _c_sp, sys as _c_sys, time as _c_time, json as _c_json
from pathlib import Path as _c_Path

_C_T_NB = NB_T0
_C_SUMMARY = {}
OURS_EXPECT_RUNS = 14                              # full-data: 8 general (seeds 42/1/2/3 + 4 recipes) + LM s42/s1/s2 + PF OA s42/s1/s2
_C_ENV_DIR = "/kaggle/working/_ours_env"
_C_WORK = _c_Path("/kaggle/working")
_C_T0 = _c_time.time()


def _c_find_dir(name):
    for pat in ["/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/", "/kaggle/input/*/*/*/*/"]:
        for d in sorted(_c_glob.glob(pat)):
            if _c_os.path.exists(_c_os.path.join(d, name)):
                return d.rstrip("/")
    return None


_c_assets = _c_find_dir("cnxt_v0_fold0.pt")
assert _c_assets, "attach goodpjw2008/rsna-knee-2-5d-convnext-reader (JPEG decoders + timm wheel)"
_c_sp.run([_c_sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", _C_ENV_DIR,
           "--find-links", _c_assets, "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)
_c_timm = [w for pat in ["/kaggle/input/*/timm-*.whl", "/kaggle/input/*/*/timm-*.whl", "/kaggle/input/*/*/*/timm-*.whl",
                         "/kaggle/input/*/*/*/*/timm-*.whl"] for w in _c_glob.glob(pat)]
if _c_timm:
    _c_sp.run([_c_sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", _C_ENV_DIR, _c_timm[0]], check=False)
print(f"our leg env: decoders from {_c_assets} | timm wheel {_c_timm[0] if _c_timm else 'none (environment timm)'}")

import torch as _c_torch
_c_free = [_c_torch.cuda.mem_get_info(i)[0] for i in range(_c_torch.cuda.device_count())]
assert _c_free, "no GPU: set the accelerator to GPU T4 x2"
_c_gpu = str(max(range(len(_c_free)), key=lambda i: _c_free[i]))
_c_env = dict(_c_os.environ, CUDA_VISIBLE_DEVICES=_c_gpu,
              OUR_LEG_EXPECT_RUNS=str(OURS_EXPECT_RUNS),
              OUR_LEG_DEADLINE=str(NB_T0 + 3.5 * 3600),          # leaves R6 at least ~5 h
              PYTHONPATH=_C_ENV_DIR + _c_os.pathsep + _c_os.environ.get("PYTHONPATH", ""))
with open(_C_WORK / "_ours.log", "w") as _c_logf:
    _c_r = _c_sp.run([_c_sys.executable, "/kaggle/working/our_leg.py"], env=_c_env, stdout=_c_logf, stderr=_c_sp.STDOUT)
print((_C_WORK / "_ours.log").read_text()[-4000:])
print(f"[our leg] rc {_c_r.returncode} in {(_c_time.time() - _C_T0) / 60:.1f} min")
assert _c_r.returncode == 0 and (_C_WORK / "_ours.csv").exists(), "our leg failed: see the log above (no fallback)"

_C_SUMMARY["ours_min"] = round((_c_time.time() - _C_T0) / 60, 1)
import gc as _c_gc; _c_gc.collect(); _c_torch.cuda.empty_cache()


In [ ]:
# ===== R6 stage, setup: public R6 code (sha-checked) + bundle fe055406 (sha-checked) -> worker arguments =====
ARM = 'E_lrprior'
BUNDLE_ID = 'fe055406fad6a12f'
DEADLINE_HOURS = 8.3
CODE_SLUG = 'rsna-orthofoundation-fold0-epoch7-20261004'
COMPETITION = 'rsna-knee-abnormality-detection'
CODE_SHA = {'native_cache.py': 'c53ce8985406b3b918d9e821d43518479e89d9f946daebd817f378e3949718b9', 'native_data.py': 'f59afe205c8e193369fa3ccd22edcce8df3b293205fc48ddb1dd45169ee43b36', 'input_provenance.py': 'b3a22500c7f976f6ee040239ada5cacecf2326861cb49af1c7d8cfdeeb55fcda', 'intensity_mapping.py': '19be2d8cbd6f8971c1288d7512a8cca41d143470c2ba6e9b9096fde035fa9d41', 'coronal_views.py': '841f216d2682ae2c0f8afd82fc902b9fa74924d7145b952069dea8b0bbd03c40', 'exact_duplicates.py': '51b38db21ce9c5fe2b6278a835983414756b61ac6af81aba380b44b41b075897', 'dicom_geometry.py': '12aa501f9704aef15b7a3bbd560951acb6d95b89b7de0141d6cf89fc3f051ab1', 'common.py': '8283fa1f8e7ef8cfacbba52df1715636be3c9ad688e39912814e7d5bcf57105f'}
SOURCE_SHA = {'px_cache.py': 'b13e38a7ad3bd496f8027e02209653e224b0926db4f0853eeb0d076a2d6e2632', 'compact_train.py': '04df7e2e154b2c2521696908f272940c63ce17e9b6337b85847fee64504eeaba', 'compact_infer.py': '8401812cd63737ec99839d317c4037c4efb445d1baec892fa7481bb30399cc2d', 'probe_infer.py': 'a6cc31448019c84be5868d0a2ff1d99719b23b0f86110efc03efff3ce6bd6980', 'probe_dual.py': 'a6155eb08701f9bb746f838bc8ef8e2129c85f76a5c58e379692ffe2e76bbef0', 'knee_anchor_bank_v2.py': 'e564807a129dfcd3c984348917436ee6083066a4d685340587ebd1d3ea14f309', 'side_common.py': 'bbcc5d027834e6a6fd48c4a8cf228e25661b7522c792587c840fe4d7cd5c0566'}
C_SLUG = 'rsna-knee-r6-frozen-fe055406-w1'   # [reweight] new private copy of the R6 bundle = version 1 of rsna-knee-r6-frozen-fe055406 (manifest sha256 6508ce6921eda986.. = 57068142's), whose latest version now holds R7
FLIP_TTA = True   # [d957] package E: v4c's flip TTA on R6
W_SLUG = 'rsna-knee-ofl-ssl-weights'
SRC_SLUG = 'rsna-knee-r6-src'   # R6's 7 worker scripts (public R6 notebook code), one .py file each; sha-checked below

import os, sys, json, shutil, subprocess, zipfile, time, hashlib
from pathlib import Path
T0 = NB_T0   # the 8.3 h deadline counts from the notebook start (our leg ran first)
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'], check=False)
subprocess.run(['df', '-h', '/dev/shm', '/kaggle/working', '/tmp'], check=False)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
TMP = Path('/kaggle/temp') if Path('/kaggle/temp').is_dir() else Path('/tmp')
TMP = TMP / 'probe'
TMP.mkdir(parents=True, exist_ok=True)

def find_mount(slug, required):
    cands = [INPUT / slug, INPUT / 'competitions' / slug]
    cands += list(INPUT.glob('datasets/*/' + slug)) + list(INPUT.glob('datasets/*/' + slug + '/*'))
    cands += list(INPUT.glob('*/' + slug)) + list(INPUT.glob(slug + '/*'))
    found = []
    for p in cands:
        if (p / required).is_file() and p not in found:
            found.append(p)
    if len(found) != 1:
        raise RuntimeError('Cannot resolve %s: %s' % (slug, found))
    return found[0]

CODE_SRC = find_mount(CODE_SLUG, 'receipt.json')
DATA = find_mount(COMPETITION, 'test.csv')

# mainline decode + rendering code: byte-identical to what built the training caches
CODE = TMP / 'mainline_code'
if (CODE_SRC / 'cloud_code.zip').is_file():
    with zipfile.ZipFile(CODE_SRC / 'cloud_code.zip') as z:
        z.extractall(CODE)
else:
    shutil.copytree(CODE_SRC / 'cloud_code', CODE, dirs_exist_ok=True)
for name, digest in CODE_SHA.items():
    assert hashlib.sha256((CODE / 'training' / name).read_bytes()).hexdigest() == digest, ('mainline module differs', name)

# pinned wheels (same versions as the runs that produced the training caches)
WHEELS = TMP / 'wheels'
WHEELS.mkdir(exist_ok=True)
if (CODE_SRC / 'dependencies.zip').is_file():
    with zipfile.ZipFile(CODE_SRC / 'dependencies.zip') as z:
        for n in z.namelist():
            if n.endswith('.whl'):
                (WHEELS / Path(n).name).write_bytes(z.read(n))
else:
    for p in CODE_SRC.rglob('*.whl'):
        shutil.copy(p, WHEELS / p.name)
wheels = sorted(str(p) for p in WHEELS.glob('*.whl'))
assert wheels, 'no wheels found'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', *wheels], check=True)

SRC = TMP / 'probe_src'
SRC.mkdir(exist_ok=True)
R6_SRC = find_mount(SRC_SLUG, 'probe_dual.py')
for name, digest in SOURCE_SHA.items():
    shutil.copy(R6_SRC / name, SRC / name)
    assert hashlib.sha256((SRC / name).read_bytes()).hexdigest() == digest, ('R6 script differs', name)
BUNDLE = TMP / 'bundle'
BUNDLE.mkdir(exist_ok=True)

M_SRC = find_mount(C_SLUG, 'manifest.json')
manifest = json.loads((M_SRC / 'manifest.json').read_text())
assert manifest['bundle_id'] == BUNDLE_ID, (manifest['bundle_id'], BUNDLE_ID)
for name in manifest['models'] + (manifest.get('models2') or []):
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
if manifest.get('state_spec'):
    name = manifest['state_spec']
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
if manifest.get('side_model'):
    name = manifest['side_model']
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
if manifest.get('fluid_stacker'):
    name = manifest['fluid_stacker']
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
if manifest.get('state_stacker'):
    name = manifest['state_stacker']
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
WORKER = 'compact_infer.py'
WORKER_ARGS = ['--backbone', manifest['backbone'], '--size', str(manifest['size']), '--crop', str(manifest['crop']),
               '--eval-stride', str(manifest.get('eval_stride', 3)), '--views', manifest.get('views', 'v1')]
if manifest.get('state_spec'):
    WORKER_ARGS += ['--state-spec', str(BUNDLE / manifest['state_spec'])]
if manifest.get('side_model'):
    # image side check: studies without one trustworthy laterality tag are re-rendered when the orientation classifier
    # says they are mirrored (same rule as the S5 training-cache fix)
    WORKER_ARGS += ['--side-model', str(BUNDLE / manifest['side_model']), '--side-threshold', str(manifest.get('side_threshold', 2.0)),
                    '--side-position-mm', str(manifest.get('side_position_mm', 20.0)), '--side-max-x', str(manifest.get('side_max_x', 0.0))]
WORKER_ARGS += ['--slices', str(manifest.get('slices', 3)), '--phys-mm', str(manifest.get('phys_mm', 0.0))]
if manifest.get('models2'):
    # second architecture on the same windows (one decode per study), per-class weights fixed in the bundle
    WORKER_ARGS += ['--models2', str(BUNDLE), '--arm2', manifest['arm2'], '--backbone2', manifest['backbone2'],
                    '--w2', ','.join(str(x) for x in manifest['w2'])]
    if manifest.get('crop2') is not None:
        # v3: the second group's own centre crop of the same decoded windows
        WORKER_ARGS += ['--crop2', str(manifest['crop2'])]
if manifest.get('trainer_md5'):
    # v3: the embedded compact_train.py must be the trainer this bundle was built for (and the worker re-checks what it imports)
    assert hashlib.md5((SRC / 'compact_train.py').read_bytes()).hexdigest() == manifest['trainer_md5'], 'pinned trainer differs from the bundle manifest (md5)'
    assert SOURCE_SHA['compact_train.py'] == manifest['src_sha256']['compact_train.py'], 'pinned trainer differs from the bundle manifest (sha256)'
    WORKER_ARGS += ['--trainer-md5', manifest['trainer_md5']]
if manifest.get('compile'):
    # v3: torch.compile of the fold models' encoders (checked against eager inside the worker; eager fallback, logged)
    WORKER_ARGS += ['--compile']
if manifest.get('fluid_stacker'):
    # v4b: D2 fluid volume measured on the rendered rows; the classes the json adopts get the stacker's logit
    WORKER_ARGS += ['--fluid-stacker', str(BUNDLE / manifest['fluid_stacker'])]
if manifest.get('flip_tta') or FLIP_TTA:
    # v4c: every fold model also scores the mirrored windows; the two passes are averaged in logit space (MCL: plain pass)
    # [d957] FLIP_TTA: this notebook's switch, at the manifest key's position (the R6 manifest predates flip TTA: no flip_tta key)
    WORKER_ARGS += ['--flip-tta']
if manifest.get('state_stacker'):
    # v4c: the classes the json adopts get b0 + coef . [z, state probabilities, binaries (, lv, miss)]
    WORKER_ARGS += ['--state-stacker', str(BUNDLE / manifest['state_stacker'])]
if manifest.get('views') in ('v3k', 'v4'):
    # knee-centred views: the anchors are rebuilt from the test DICOM headers + the SKM-TEA checkpoint of the code dataset
    assert hashlib.sha256((CODE_SRC / 'anatomy.pt').read_bytes()).hexdigest() == manifest['anchor_weights_sha256'], 'anatomy.pt differs from the training anchor checkpoint'
    WORKER_ARGS += ['--anchor-weights', str(CODE_SRC / 'anatomy.pt')]
if manifest['backbone'] == 'ofl_l':
    # the fold models hold only the trained tail; rebuild the frozen part from the public OFL-L weights
    W_SRC = find_mount(W_SLUG, 'weight_parts.json')
    parts = json.loads((W_SRC / 'weight_parts.json').read_text())
    h = hashlib.sha256()
    with (BUNDLE / 'model.safetensors').open('wb') as dst:
        for item in parts['parts']:
            blob = (W_SRC / item['name']).read_bytes()
            assert len(blob) == item['bytes'] and hashlib.sha256(blob).hexdigest() == item['sha256'], item['name']
            dst.write(blob)
            h.update(blob)
    assert h.hexdigest() == parts['assembled_sha256'] == manifest['init_weights_sha256'], 'encoder weights differ'
    del blob
    WORKER_ARGS += ['--init', str(BUNDLE / 'model.safetensors'), '--freeze-blocks', str(manifest['freeze_blocks'])]
print('ASSETS_READY', str(CODE_SRC), str(M_SRC), str(DATA), 'flip_tta', '--flip-tta' in WORKER_ARGS, 'seconds %.1f' % (time.time() - T0), flush=True)


In [ ]:
# ===== R6 stage, run: probe_dual.py scores every test study -> r6_sub.csv =====
deadline = T0 + DEADLINE_HOURS * 3600
env = dict(os.environ, PYTHONPATH=str(CODE) + os.pathsep + os.environ.get('PYTHONPATH', ''), PYTHONDONTWRITEBYTECODE='1')
pred = TMP / 'prediction'
rc = subprocess.run([sys.executable, '-u', str(SRC / 'probe_dual.py'), '--data', str(DATA), '--split', 'test',
                     '--code', str(CODE), '--bundle', str(BUNDLE), '--arm', ARM, '--output', str(pred),
                     '--amp', 'auto', '--loader-workers', '2', '--deadline', str(deadline),
                     '--worker', WORKER, '--worker-args', json.dumps(WORKER_ARGS),
                     '--max-fallback-fraction', '1.0'], env=env).returncode   # no 3 % stop here: the blend checks R6's coverage
for log in sorted(pred.glob('worker*/log.txt')):
    shutil.copyfile(log, WORK / (log.parent.name + '_log.txt'))
if rc != 0:
    raise RuntimeError('controller failed with exit code %d' % rc)
shutil.copyfile(pred / 'submission.csv', WORK / 'r6_sub.csv')

receipt = json.loads((pred / 'submission_receipt.json').read_text())
receipt['seconds_including_notebook_setup'] = time.time() - T0
receipt['bundle_id'] = BUNDLE_ID
receipt['arm'] = ARM
(WORK / 'submission_receipt.json').write_text(json.dumps(receipt, indent=1))
print('OURS_STAGE_COMPLETE', json.dumps({k: receipt.get(k) for k in ('status', 'arm', 'rows', 'scored', 'fallback_studies',
                                                                 'seconds', 'per_study_mean', 'gpus')}), flush=True)
import pandas as pd
display(pd.read_csv(WORK / 'r6_sub.csv').head())


In [ ]:
# ===== Final blend: our 14-model leg + R6 -> submission.csv =====
# Per finding: final = wo * ours + (1 - wo) * R6, in rank space. wo = 0.55 by default, with the public base's per-finding
# Raptor-vs-R6 pattern for four findings (our leg takes Raptor's role); our mean share is 0.53. Public LB .956.
# All parts are percentile ranks over the studies each part scored; a missing part is dropped and the rest renormalised per study.
import numpy as _f_np, pandas as _f_pd
from scipy.stats import rankdata as _f_rankdata

W_OURS = 0.55
W_OURS_C = {'Lateral OA': 0.2, 'PF OA': 0.6, 'Synovitis': 0.8, "Baker's": 0.35}   # every other finding: W_OURS

try:
    _f_root = next(_c_Path(d) for pat in ["/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"]
                   for d in sorted(_c_glob.glob(pat))
                   if _c_os.path.exists(_c_os.path.join(d, "test.csv")) and _c_os.path.exists(_c_os.path.join(d, "test_series.csv")))
    _f_ids = _f_pd.read_csv(_f_root / "test.csv", dtype={"StudyInstanceUID": str})["StudyInstanceUID"].tolist()
    _f_labels = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
                 "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
    n = len(_f_ids)

    def _f_load(path):
        if not path.exists():
            print(f"[blend] {path.name}: missing"); return None
        d = _f_pd.read_csv(path, dtype={"StudyInstanceUID": str}).drop_duplicates("StudyInstanceUID")
        d = d.set_index("StudyInstanceUID").reindex(_f_ids)[_f_labels]
        ok = d.notna().all(axis=1).to_numpy().copy()
        if ok.mean() < 0.97:
            print(f"[blend] {path.name}: dropped (coverage {ok.mean():.1%})"); return None
        R = _f_np.full((n, len(_f_labels)), _f_np.nan)
        for j, col in enumerate(_f_labels):
            R[ok, j] = _f_rankdata(d[col].to_numpy()[ok]) / ok.sum()
        print(f"[blend] {path.name}: {ok.sum():,}/{n:,} studies")
        return R

    def _f_wavg(parts):
        num = _f_np.zeros((n, len(_f_labels))); den = _f_np.zeros((n, len(_f_labels)))
        for R, w in parts:
            if R is None:
                continue
            ok = _f_np.isfinite(R)
            num += w[None, :] * _f_np.where(ok, R, 0); den += w[None, :] * ok
        return _f_np.where(den > 0, num / _f_np.maximum(den, 1e-9), _f_np.nan)

    R6 = _f_load(_C_WORK / "r6_sub.csv")
    OURS = _f_load(_C_WORK / "_ours.csv")
    assert OURS is not None, "our leg output missing: not submitting an R6-only file under this name"
    assert R6 is not None, "R6 output missing: not submitting an ours-only file under this name"
    wo = _f_np.array([W_OURS_C.get(col, W_OURS) for col in _f_labels])
    B = _f_wavg([(OURS, wo), (R6, 1 - wo)])
    B = _f_np.where(_f_np.isfinite(B), B, 0.5)
    _f_out = _f_pd.DataFrame(B, columns=_f_labels)
    for col in _f_labels:
        _f_out[col] = _f_rankdata(_f_out[col]) / len(_f_out)
    _f_out.insert(0, "StudyInstanceUID", _f_ids)
    if n > 2:
        rho = [_f_np.corrcoef(_f_np.nan_to_num(R6[:, j], nan=.5), _f_np.nan_to_num(OURS[:, j], nan=.5))[0, 1] for j in range(len(_f_labels))]
        print(f"[blend] rank correlation R6 vs ours: mean {_f_np.nanmean(rho):.2f}")
    assert len(_f_out) == n and _f_np.isfinite(_f_out[_f_labels].to_numpy()).all()
    _f_tmp = _C_WORK / "_submission_tmp.csv"
    _f_out.to_csv(_f_tmp, index=False)
    _c_os.replace(_f_tmp, _C_WORK / "submission.csv")
    _C_SUMMARY["blend"] = {"mode": "ours+R6", "ours_w": dict(zip(_f_labels, wo.tolist())), "ours_mean_share": round(float(wo.mean()), 3)}
    print(f"[blend] wrote submission.csv {_f_out.shape} | {_C_SUMMARY['blend']}")
except Exception as _f_e:
    _C_SUMMARY["blend"] = f"failed ({type(_f_e).__name__}: {_f_e})"
    print(f"[blend] FAILED ({type(_f_e).__name__}: {_f_e})")
    raise
_C_SUMMARY["total_h"] = round((_c_time.time() - _C_T_NB) / 3600, 3)
(_C_WORK / "_runtime.json").write_text(_c_json.dumps(_C_SUMMARY, indent=1, default=str))
print("[runtime]", _c_json.dumps(_C_SUMMARY, default=str))
